In [ ]:
from tensorflow.python.summary.summary_iterator import summary_iterator
import pandas as pd
import os
from pathlib import Path


def load_tensorboard_data(logdir):
    data = []
    for root, _, files in os.walk(logdir):
        for file in files:
            if "events.out.tfevents" in file:
                filepath = os.path.join(root, file)
                for summary in summary_iterator(filepath):
                    file = Path(filepath)
                    metric = file.parent.name

                    run = file.parents[2].name + "/" + file.parents[1].name

                    for value in summary.summary.value:
                        if value.HasField("simple_value"):
                            data.append(
                                {
                                    "run": run,
                                    "metric": metric,
                                    "wall_time": summary.wall_time,
                                    "step": summary.step,
                                    "tag": value.tag,
                                    "value": value.simple_value,
                                }
                            )
    df = pd.DataFrame(data)
    return df


df = load_tensorboard_data("./colony_segmentation/runs")

# Show every unique value from the tag column
print(df["metric"].unique())
print(df["run"].unique())

In [ ]:
random_init_data = df[df["run"] == "2024-06-05_15-52/final"]
coco_noagar_data = df[df["run"] == "2024-06-05_15-54/final"]
random_agar_data = df[df["run"] == "2024-06-05_16-09/final"]
coc_agar_data = df[df["run"] == "2024-06-05_16-12/final"]

In [ ]:
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "nature"])

fig, ax = plt.subplots(1, 1, figsize=(3, 2))

data = random_init_data[random_init_data["metric"] == "val_loss_total"]
ax.plot(data["step"], data["value"], label="Random")
data = coco_noagar_data[coco_noagar_data["metric"] == "val_loss_total"]
ax.plot(data["step"], data["value"], label="COCO")
data = random_agar_data[random_agar_data["metric"] == "val_loss_total"]
ax.plot(data["step"], data["value"], label="Random+AGAR")
data = coc_agar_data[coc_agar_data["metric"] == "val_loss_total"]
ax.plot(data["step"], data["value"], label="COCO+AGAR")

ax.set_ylim(0.1, 0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig("chart_dataset_comparison_loss.png", dpi=500)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(4, 2))

data = random_init_data[random_init_data["metric"] == "val_metrics_smape"]
ax[0].plot(data["step"], data["value"], label="Random")
data = coco_noagar_data[coco_noagar_data["metric"] == "val_metrics_smape"]
ax[0].plot(data["step"], data["value"], label="COCO")
data = random_agar_data[random_agar_data["metric"] == "val_metrics_smape"]
ax[0].plot(data["step"], data["value"], label="Random+AGAR")
data = coc_agar_data[coc_agar_data["metric"] == "val_metrics_smape"]
ax[0].plot(data["step"], data["value"], label="COCO+AGAR")

ax[0].set_ylim(0.1, 0.6)
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("sMAPE")
ax[0].legend()
ax[0].grid(True)


data = random_init_data[
    random_init_data["metric"] == "val_metrics_avg_precision_iou_50"
]
ax[1].plot(data["step"], data["value"], label="Random")
data = coco_noagar_data[
    coco_noagar_data["metric"] == "val_metrics_avg_precision_iou_50"
]
ax[1].plot(data["step"], data["value"], label="COCO")
data = random_agar_data[
    random_agar_data["metric"] == "val_metrics_avg_precision_iou_50"
]
ax[1].plot(data["step"], data["value"], label="Random+AGAR")
data = coc_agar_data[coc_agar_data["metric"] == "val_metrics_avg_precision_iou_50"]
ax[1].plot(data["step"], data["value"], label="COCO+AGAR")

ax[1].set_ylim(0.3, 0.85)
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("mAP@0.5:0.95:0.05")
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.savefig("chart_dataset_comparison.png", dpi=500)
plt.show()